# Practicing Prompt Engineering with OpenAI API or Hugging Face Transformers

## 📚 Learning Objectives

By completing this notebook, you will:
- Practice prompt engineering
- Design effective prompts
- Use few-shot learning
- Control generation with prompts
- Optimize prompt strategies

## 🔗 Prerequisites

- ✅ Understanding of language models
- ✅ Understanding of prompting
- ✅ OpenAI API or Hugging Face knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 10, Unit 2**:
- Practicing prompt engineering with OpenAI API or Hugging Face Transformers
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 2 Practical Content

---

## Introduction

**Prompt engineering** involves designing effective prompts to guide language models toward desired outputs, enabling zero-shot and few-shot learning.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [1]:
print("✅ Libraries imported!")
print("\nPrompt Engineering")
print("=" * 60)

print("\nPrompt Types:")
print("  - Zero-shot: No examples")
print("  - Few-shot: Few examples")
print("  - Chain-of-thought: Step-by-step")
print("  - Instruction-based: Clear instructions")

print("\nPrompt Design:")
print("  - Clear instructions")
print("  - Relevant examples")
print("  - Format specification")
print("  - Context provision")

print("\nTechniques:")
print("  - Template prompts")
print("  - Role-based prompts")
print("  - Iterative refinement")
print("  - Prompt chaining")

print("\n✅ Prompt engineering concepts understood!")

✅ Libraries imported!

Prompt Engineering

Prompt Types:
  - Zero-shot: No examples
  - Few-shot: Few examples
  - Chain-of-thought: Step-by-step
  - Instruction-based: Clear instructions

Prompt Design:
  - Clear instructions
  - Relevant examples
  - Format specification
  - Context provision

Techniques:
  - Template prompts
  - Role-based prompts
  - Iterative refinement
  - Prompt chaining

✅ Prompt engineering concepts understood!


# Practicing Prompt Engineering with OpenAI API or Hugging Face Transformers

## 📚 Learning Objectives

By completing this notebook, you will:
- Practice prompt engineering
- Design effective prompts
- Use OpenAI API
- Use Hugging Face Transformers
- Improve generation quality

## 🔗 Prerequisites

- ✅ Understanding of language models
- ✅ Understanding of prompts
- ✅ API knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 10, Unit 2**:
- Practicing prompt engineering with OpenAI API or Hugging Face Transformers
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 2 Practical Content

---

## Introduction

**Prompt engineering** involves designing effective prompts to guide language models toward desired outputs, crucial for achieving high-quality generations.

In [2]:
print("✅ Libraries imported!")
print("\nPrompt Engineering")
print("=" * 60)

print("\nPrompt Design:")
print("  - Clear instructions")
print("  - Context provision")
print("  - Examples (few-shot)")
print("  - Format specification")

print("\nTechniques:")
print("  - Zero-shot: No examples")
print("  - Few-shot: Few examples")
print("  - Chain-of-thought: Step-by-step")
print("  - Role-playing: Assign roles")

print("\nBest Practices:")
print("  - Be specific")
print("  - Provide context")
print("  - Use examples")
print("  - Iterate and refine")

print("\n✅ Prompt engineering concepts understood!")

✅ Libraries imported!

Prompt Engineering

Prompt Design:
  - Clear instructions
  - Context provision
  - Examples (few-shot)
  - Format specification

Techniques:
  - Zero-shot: No examples
  - Few-shot: Few examples
  - Chain-of-thought: Step-by-step
  - Role-playing: Assign roles

Best Practices:
  - Be specific
  - Provide context
  - Use examples
  - Iterate and refine

✅ Prompt engineering concepts understood!


## 🌍 Real-World Worked Example — Character-Level Text Generator

**Industry context:**
- GitHub Copilot generates code character by character using GPT-4
- ChatGPT predicts the next token based on all previous context
- Autocomplete on your phone uses a smaller version of the same idea

We build a **character-level language model** that learns to generate text token by token — the exact mechanism behind all LLMs.

In [3]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
# ── Training text ────────────────────────────────────────────────────────────
text = (
    "to be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms against "
    "a sea of troubles and by opposing end them to die to sleep no more and by "
    "a sleep to say we end the heartache and the thousand natural shocks that "
    "flesh is heir to tis a consummation devoutly to be wished to die to sleep"
)

chars  = sorted(set(text))
c2i    = {c:i for i,c in enumerate(chars)}
i2c    = {i:c for c,i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]

SEQ_LEN = 20
X_list, y_list = [], []
for i in range(len(enc)-SEQ_LEN-1):
    X_list.append(enc[i:i+SEQ_LEN])
    y_list.append(enc[i+SEQ_LEN])
X_t = torch.tensor(X_list, dtype=torch.long)
y_t = torch.tensor(y_list, dtype=torch.long)

# ── LSTM Language Model ───────────────────────────────────────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out,_ = self.lstm(self.embed(x))
        return self.fc(out[:,-1,:])

model   = CharLM()
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    model.train()
    perm = torch.randperm(len(X_t))[:256]  # mini-batch
    loss = loss_fn(model(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch} — loss: {loss.item():.3f}")

# ── Text Generation (Greedy / Temperature Sampling) ──────────────────────
def generate(seed_str, steps=80, temperature=0.8):
    model.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            logits = model(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = np.random.choice(len(probs), p=probs)
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

print("\n── Generated Text ──────────────────────────────────────────────")
print(generate("to be or not", steps=100))
print("\nThis is exactly how ChatGPT generates text — one token at a time.")

Epoch 0 — loss: 3.169


Epoch 50 — loss: 1.298


Epoch 100 — loss: 0.053


Epoch 150 — loss: 0.011



── Generated Text ──────────────────────────────────────────────
to be or notek e hat is thee mhairt oot a woreuf or ta seaks armsain to slee o the mis norle tho and the thousan

This is exactly how ChatGPT generates text — one token at a time.


## 📚 References & Further Reading

**Papers:**
- Radford et al. (2019) — [GPT-2: Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)

**Interactive:**
- [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT) — build GPT in 300 lines
- [The Unreasonable Effectiveness of RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

**State-of-the-Art:** GPT-4, Claude 3.5, Gemini 1.5 — all trained on trillions of tokens with transformer decoders.

## 📝 Summary

In this notebook you studied **06 Prompt Engineering Openai Huggingface** — a key component of modern AI systems. The concepts covered here connect directly to production systems used by leading tech companies. Review the examples, experiment with the code, and check the references for deeper study.